# Week 3 demo · Satellite image processing on a real Sentinel-2 scene
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/trongan93/dl-space-2026/blob/main/notebooks/Week3_Demo_Satellite_Image_Processing.ipynb)  ·  repo: [trongan93/dl-space-2026](https://github.com/trongan93/dl-space-2026)

**Deep Learning in Space Technology Applications · 115-1 · NTUT · Dr. Trong-An Bui**

This notebook is the live companion of the Week 3 slides. Every section number below is printed on the matching slide.
Run it top to bottom in Colab (Runtime → Run all, ~3 minutes). It will

1. fetch one **real** Sentinel-2 L2A window over Taipei (or fall back to the course sample / a synthetic scene),
2. open the GeoTIFF the way the slides describe it — header, IFD, geo-tags, nodata, bands,
3. run the classical toolbox on it: stretch, histograms, convolution, edges, Otsu, morphology,
4. end with **Class work 3** and a preview of Lab 1's chips and splits.

> If you have your own Lab 0 output (`lab0_cube.tif`, `lab0_scl.npy`), upload both to the session first — the notebook prefers your scene.

In [ ]:
# --- 0 · environment (Colab: ~1 minute the first time) ---
import importlib, subprocess, sys
for pkg, mod in [("rasterio", "rasterio"), ("pystac-client", "pystac_client"), ("scikit-image", "skimage"), ("opencv-python-headless", "cv2")]:
    try: importlib.import_module(mod)
    except ImportError: subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=True)
import os, json, time, urllib.request, numpy as np, matplotlib.pyplot as plt, rasterio, cv2, skimage
from rasterio.windows import from_bounds
from rasterio.enums import Resampling
from rasterio.warp import transform_bounds
from rasterio.transform import from_origin
print("python", sys.version.split()[0], "| numpy", np.__version__, "| rasterio", rasterio.__version__,
      "| GDAL", rasterio.__gdal_version__, "| OpenCV", cv2.__version__, "| scikit-image", skimage.__version__)

## 1 · A real scene, in one cell  *(slides 9, 18–19)*
Tier 0 = your Lab 0 files · tier 1 = live STAC search of Earth Search · tier 2 = the course sample in the repo · tier 3 = synthetic.
Read the printout: which tier did you get, what is the scene ID, and — if live — which **processing baseline**?

In [ ]:
# --- 1 · get a real scene: four tiers, first one that works wins ---
REPO_RAW   = "https://raw.githubusercontent.com/trongan93/dl-space-2026/main/data/"
BBOX       = [121.35, 25.00, 121.65, 25.20]   # Tamsui river mouth -> Taipei basin (WGS84)
DATE_RANGE = "2026-01-01/2026-09-20"
MAX_CLOUD  = 20
GSD        = 10
MAX_PX     = 800                          # window cap (rows/cols) so Colab RAM is safe
BANDS      = {"blue": "B02", "green": "B03", "red": "B04", "nir": "B08", "swir16": "B11", "swir22": "B12"}
BAND_ORDER = list(BANDS); I = {b: i for i, b in enumerate(BAND_ORDER)}
SCL_VALID  = [4, 5, 6, 7, 11]
SCL_NAMES  = {0: "nodata", 1: "saturated", 2: "dark", 3: "cloud shadow", 4: "vegetation", 5: "not vegetated",
              6: "water", 7: "unclassified", 8: "cloud medium", 9: "cloud high", 10: "thin cirrus", 11: "snow/ice"}
FORCE_TIER = os.environ.get("DL_SPACE_FORCE_TIER")   # "live" | "sample" | "synthetic" (testing only)

def tier_upload():
    """Tier 0: the student's own Lab 0 output in the working directory."""
    if FORCE_TIER or not (os.path.exists("lab0_cube.tif") and os.path.exists("lab0_scl.npy")):
        raise FileNotFoundError("no lab0_cube.tif / lab0_scl.npy here")
    with rasterio.open("lab0_cube.tif") as src:
        cube, crs, tr, tags = src.read().astype("float32"), src.crs, src.transform, src.tags()
    return cube, np.load("lab0_scl.npy"), crs, tr, tags.get("scene_id", "lab0_cube.tif (your Lab 0)"), "upload"

def tier_live():
    """Tier 1: search Earth Search (STAC) and read one window from the cloud-optimised GeoTIFFs."""
    if FORCE_TIER not in (None, "live"): raise RuntimeError("tier skipped")
    from pystac_client import Client
    cat = Client.open("https://earth-search.aws.element84.com/v1")
    items = list(cat.search(collections=["sentinel-2-l2a"], bbox=BBOX, datetime=DATE_RANGE,
                            query={"eo:cloud_cover": {"lt": MAX_CLOUD}}, max_items=40).item_collection())
    if not items: raise RuntimeError("no scene matched")
    items.sort(key=lambda i: i.properties["eo:cloud_cover"]); item = items[0]
    with rasterio.open(item.assets["red"].href) as src: crs = src.crs
    b = transform_bounds("EPSG:4326", crs, *BBOX)
    W = min(int(round((b[2] - b[0]) / GSD)), MAX_PX); H = min(int(round((b[3] - b[1]) / GSD)), MAX_PX)
    b = (b[0], b[3] - H * GSD, b[0] + W * GSD, b[3])
    def read(href, resampling):
        with rasterio.open(href) as src:
            win = from_bounds(*b, transform=src.transform)
            return src.read(1, window=win, out_shape=(H, W), resampling=resampling, boundless=True, fill_value=0)
    layers = []
    for k in BAND_ORDER:
        dn = read(item.assets[k].href, Resampling.bilinear)
        rho = (dn.astype("float32") - 1000) / 10000; rho[dn == 0] = np.nan; layers.append(rho)
    scl = read(item.assets["scl"].href, Resampling.nearest).astype("uint8")
    tags = dict(datetime=item.properties["datetime"], cloud=item.properties["eo:cloud_cover"],
                pb=item.properties.get("s2:processing_baseline", "?"), tile=item.properties.get("grid:code", "?"))
    return np.stack(layers), scl, crs, from_origin(b[0], b[3], GSD, GSD), item.id, ("live", tags)

def tier_sample():
    """Tier 2: the course's bundled real Sentinel-2 window (data/ in the repo, or downloaded from GitHub)."""
    if FORCE_TIER not in (None, "sample"): raise RuntimeError("tier skipped")
    paths = {}
    for fn in ["taipei_s2_sample.tif", "taipei_s2_sample_scl.tif"]:
        for cand in [fn, os.path.join("data", fn), os.path.join("..", "data", fn)]:
            if os.path.exists(cand): paths[fn] = cand; break
        else:
            urllib.request.urlretrieve(REPO_RAW + fn, fn); paths[fn] = fn
    with rasterio.open(paths["taipei_s2_sample.tif"]) as src:
        cube, crs, tr, tags = src.read().astype("float32"), src.crs, src.transform, src.tags()
    with rasterio.open(paths["taipei_s2_sample_scl.tif"]) as src: scl = src.read(1).astype("uint8")
    return cube, scl, crs, tr, tags.get("scene_id", "bundled sample"), "sample"

def tier_synthetic():
    """Tier 3: a labelled synthetic scene so every cell still runs. NOT real data."""
    rng = np.random.default_rng(0); H, W = 600, 800
    yy, xx = np.mgrid[0:H, 0:W] / 100.0
    river = np.abs(yy - 3 - 1.2 * np.sin(xx / 1.5)) < 0.25; sea = xx < 1.2 + 0.3 * np.sin(yy)
    lake = ((xx - 6.5) ** 2 + (yy - 4.8) ** 2) < 0.35
    water = river | sea | lake; urban = (xx > 3.5) & (yy > 3.5) & ~water
    def band(wv, vv, uv, n=0.02):
        a = np.where(water, wv, np.where(urban, uv, vv)).astype("float32"); return np.clip(a + rng.normal(0, n, a.shape), 0.001, 0.95)
    cube = np.stack([band(.06,.04,.12), band(.08,.07,.14), band(.05,.05,.16), band(.02,.45,.22), band(.01,.25,.30), band(.005,.15,.28)])
    cloud = ((xx - 6) ** 2 + (yy - 1.5) ** 2) < 0.6; cube[:, cloud] = 0.7
    scl = np.where(water, 6, np.where(urban, 5, 4)).astype("uint8"); scl[cloud] = 9
    cube[:, :15, :] = np.nan; scl[:15, :] = 0
    return cube, scl, rasterio.crs.CRS.from_epsg(32651), from_origin(290000, 2790000, GSD, GSD), "SYNTHETIC-SCENE - not real data", "synthetic"

t0 = time.time(); SOURCE = None
for fn in (tier_upload, tier_live, tier_sample, tier_synthetic):
    try:
        cube, scl, cube_crs, cube_transform, SCENE_ID, SOURCE = fn(); break
    except Exception as e:
        print(f"  {fn.__name__:15s} -> skipped: {repr(e)[:90]}")
if isinstance(SOURCE, tuple): SOURCE, live_tags = SOURCE; print("  live scene:", live_tags)
valid = np.isin(scl, SCL_VALID) & np.isfinite(cube).all(0)
print(f"\nSCENE: {SCENE_ID}\nsource tier: {SOURCE} | cube (C,H,W) = {cube.shape} {cube.dtype} | CRS {cube_crs} | {time.time()-t0:.1f} s")
print(f"reflectance range (valid): {np.nanmin(cube[:, valid]):.3f} - {np.nanmax(cube[:, valid]):.3f} | valid fraction {100*valid.mean():.1f} %")
if SOURCE == "synthetic": print("\n>>> SYNTHETIC scene: fine for following the demo, NOT valid for any submission.")

## 2 · Inside the GeoTIFF  *(slide 16: header · image data · IFD · geo-tags · metadata · bands)*
We write the cube to disk once, then read it back **byte by byte**: the 8-byte header first, then what `rasterio` decodes from the IFD and the GeoTIFF tags.

In [ ]:
tif = "demo_cube.tif"
profile = dict(driver="GTiff", dtype="float32", count=cube.shape[0], height=cube.shape[1], width=cube.shape[2],
               crs=cube_crs, transform=cube_transform, nodata=np.nan, compress="deflate", tiled=True, blockxsize=256, blockysize=256)
with rasterio.open(tif, "w", **profile) as dst:
    dst.write(cube); dst.descriptions = tuple(f"{k} {BANDS[k]}" for k in BAND_ORDER)
    dst.update_tags(scene_id=SCENE_ID, decode="rho=(DN-1000)/10000", source_tier=SOURCE)
    dst.build_overviews([2, 4, 8], Resampling.average)          # -> a cloud-optimised layout

# 1 Header: 8 raw bytes
hdr = open(tif, "rb").read(8)
order = {b"II": "little-endian (Intel)", b"MM": "big-endian (Motorola)"}[hdr[:2]]
magic = int.from_bytes(hdr[2:4], "little" if hdr[:2] == b"II" else "big")
ifd0  = int.from_bytes(hdr[4:8], "little" if hdr[:2] == b"II" else "big")
print(f"1 HEADER   bytes {hdr!r} -> byte order {order}, magic {magic} ({'classic TIFF' if magic == 42 else 'BigTIFF'}), first IFD at byte {ifd0}")

with rasterio.open(tif) as src:
    print(f"2 IMAGE DATA   {src.width} x {src.height} px, {src.dtypes[0]}, compression {src.compression}, tiled {src.is_tiled} {src.block_shapes[0]}")
    print(f"3 IFD TAGS     count={src.count} width={src.width} height={src.height} dtype={src.dtypes[0]} interleave={src.interleaving}")
    print(f"4 GEO TAGS     CRS {src.crs}  |  transform {tuple(round(v, 1) for v in src.transform[:6])}  |  pixel {src.res} m")
    e, n = src.xy(0, 0); print(f"               pixel (0,0) centre = E {e:,.0f} m, N {n:,.0f} m;  bounds {tuple(round(v) for v in src.bounds)}")
    print(f"5 METADATA     nodata={src.nodata}  tags={src.tags()}")
    print(f"6 BANDS        {src.descriptions}")
    print(f"COG extras     overviews for band 1: {src.overviews(1)}  (read a 1/4 view: src.read(1, out_shape=(H//4, W//4)))")
    print(f"               a true COG (rio cogeo create) also moves every IFD to the front of the file, so a client needs one small read to plan its range requests")
print("file size %.1f MB" % (os.path.getsize(tif) / 1e6))

The same information from the command line — `rio info` ships with rasterio; on a machine with GDAL ≥ 3.11 you would type `gdal raster info demo_cube.tif` *(slide 17)*.

In [ ]:
!rio info --indent 1 demo_cube.tif | head -40

## 3 · The image is a matrix  *(slide 14)*
Index a band, slice a window, vectorise the arithmetic. Print `.shape` before every operation.

In [ ]:
print("cube", cube.shape, cube.dtype, "| one band", cube[I["red"]].shape, "| a 256-px chip", cube[:, 256:512, 0:256].shape)
red, nir, green = cube[I["red"]], cube[I["nir"]], cube[I["green"]]
ndvi = (nir - red) / (nir + red + 1e-6)
ndwi = (green - nir) / (green + nir + 1e-6)
water_rule = ndwi > 0                                    # a boolean mask is also an array
print("NDVI range %.2f..%.2f | NDWI>0 covers %.1f %% of valid pixels" % (np.nanmin(ndvi[valid]), np.nanmax(ndvi[valid]), 100 * water_rule[valid].mean()))
hwc = np.moveaxis(cube, 0, -1); print("(C,H,W) -> (H,W,C) for matplotlib/OpenCV:", hwc.shape)
r0, c0 = cube.shape[1] // 2, cube.shape[2] // 2
print("8 x 8 reflectance values of the red band at the centre (x1000):\n", np.round(1000 * red[r0:r0+8, c0:c0+8]).astype(int))

## 4 · Reading and visualising: why the raw composite is dark, and what a stretch does  *(slides 19–20)*
Reflectance of land is 0.05–0.3; the screen expects 0–1. The fix is **display-side**: percentile stretch or CLAHE. The model never sees it.

In [ ]:
from skimage import exposure
def stretch(img, lo=2, hi=98):
    out = np.empty_like(img)
    for i in range(img.shape[-1]):
        a, b = np.nanpercentile(img[..., i], [lo, hi]); out[..., i] = np.clip((img[..., i] - a) / (b - a + 1e-9), 0, 1)
    return np.nan_to_num(out)
rgb = np.stack([cube[I["red"]], cube[I["green"]], cube[I["blue"]]], -1)
raw = np.nan_to_num(np.clip(rgb, 0, 1)); st = stretch(rgb); clahe = exposure.equalize_adapthist(st, clip_limit=0.03)
fig, ax = plt.subplots(1, 3, figsize=(16, 5.5))
for a, im, t in zip(ax, [raw, st, clahe], ["raw reflectance -> dark", "2-98 % percentile stretch", "CLAHE (adaptive equalisation)"]):
    a.imshow(im); a.set_title(t); a.axis("off")
fig.suptitle(f"{SCENE_ID}  |  {cube.shape[2]} x {cube.shape[1]} px at {GSD} m", y=1.0); plt.tight_layout(); plt.show()
print("mean of raw red band: %.3f  -> a screen wants ~0.5. Enhancement is for eyes, the model gets the reflectance." % np.nanmean(red))

## 5 · Histograms: the cheapest diagnostic  *(slide 21)*
Per band, then per **class** using the Sen2Cor SCL map as a rough label. Two humps → a threshold will work; one hump → it will not.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(15, 4.5))
for k in BAND_ORDER:
    ax[0].hist(cube[I[k]][valid], bins=120, range=(0, 0.6), histtype="step", label=k)
ax[0].set_title("per band (valid pixels)"); ax[0].legend(); ax[0].set_xlabel("reflectance")
for cls, col in [(6, "tab:blue"), (4, "tab:green"), (5, "tab:orange"), (9, "grey")]:
    m = scl == cls
    if m.sum() > 50: ax[1].hist(nir[m], bins=120, range=(0, 0.7), histtype="stepfilled", alpha=0.4, color=col, label=f"{SCL_NAMES[cls]} ({m.sum():,} px)")
ax[1].set_title("NIR by SCL class - is water separable?"); ax[1].legend(); ax[1].set_xlabel("NIR reflectance")
plt.tight_layout(); plt.show()
print("class shares:", {SCL_NAMES[k]: f"{100*(scl==k).mean():.1f} %" for k in np.unique(scl) if (scl==k).mean() > 0.005})

## 6 · At the door of OpenCV: dtype and axis order  *(slide 22)*
OpenCV wants `uint8`, `(H, W, C)`, BGR. Convert on purpose; keep a `float32` copy for the model.

In [ ]:
def to_uint8(band):
    a, b = np.nanpercentile(band[valid], [2, 98]); return np.clip(255 * (np.nan_to_num(band) - a) / (b - a + 1e-9), 0, 255).astype("uint8")
nir8, red8, blue8 = to_uint8(nir), to_uint8(red), to_uint8(cube[I["blue"]])
bgr = cv2.merge([blue8, to_uint8(green), red8])          # OpenCV order
rgb8 = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)              # before imshow
print("float32 band", nir.dtype, nir.shape, "->", nir8.dtype, nir8.shape, "| bgr", bgr.shape, bgr.dtype)
print("uint16 trap: cv2 would silently clip DN 12000 ->", np.uint8(np.uint16(12000)), "if you cast without scaling")

## 7 · Convolution  *(slides 23–24, Exercise 2)*
One 3×3 kernel, slid over the band: blur, sharpen, gradient. You type the nine numbers tonight; a CNN learns them.

In [ ]:
K = {"box blur 3x3": np.ones((3, 3), "float32") / 9,
     "sharpen":      np.array([[0, -1, 0], [-1, 5, -1], [0, -1, 0]], "float32"),
     "Sobel-x":      np.array([[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]], "float32"),
     "Laplacian":    np.array([[0, 1, 0], [1, -4, 1], [0, 1, 0]], "float32")}
r0, c0 = cube.shape[1] // 2 - 150, cube.shape[2] // 2 - 200          # a 300 x 400 detail window
fig, ax = plt.subplots(1, 5, figsize=(20, 4.2))
ax[0].imshow(nir8[r0:r0+300, c0:c0+400], cmap="gray"); ax[0].set_title("NIR (uint8)")
for a, (name, k) in zip(ax[1:], K.items()):
    out = cv2.filter2D(nir8.astype("float32"), -1, k)
    a.imshow(np.abs(out[r0:r0+300, c0:c0+400]) if "Sobel" in name or "Lap" in name else out[r0:r0+300, c0:c0+400], cmap="gray"); a.set_title(name)
for a in ax: a.axis("off")
plt.tight_layout(); plt.show()
print("sharpen kernel:\n", K["sharpen"], "\nsum of weights = %.0f (keeps brightness); Sobel weights sum to 0 (responds only to change)" % K["sharpen"].sum())

## 8 · Edges: Sobel then Canny  *(slide 26)*
Sobel = two convolutions → gradient magnitude. Canny = blur + Sobel + non-maximum suppression + two thresholds + hysteresis. The two numbers are a contract.

In [ ]:
gx = cv2.Sobel(nir8, cv2.CV_64F, 1, 0, ksize=3); gy = cv2.Sobel(nir8, cv2.CV_64F, 0, 1, ksize=3)
mag = np.hypot(gx, gy)
fig, ax = plt.subplots(1, 3, figsize=(17, 5))
ax[0].imshow(mag[r0:r0+300, c0:c0+400], cmap="gray"); ax[0].set_title("Sobel magnitude - every texture is an edge")
for a, (lo, hi) in zip(ax[1:], [(100, 200), (30, 90)]):
    e = cv2.Canny(nir8, lo, hi); a.imshow(e[r0:r0+300, c0:c0+400], cmap="gray"); a.set_title(f"Canny({lo}, {hi}): {100*e.mean()/255:.1f} % edge pixels")
for a in ax: a.axis("off")
plt.tight_layout(); plt.show()

## 9 · Thresholding and Otsu  *(slide 27)*
`I(x, y) > T` is a one-parameter classifier. Otsu picks `T` from the histogram (minimum within-class variance). We do it on NDWI for water and on blue for cloud — and the result is **a label**.

In [ ]:
from skimage.filters import threshold_otsu
T_w = threshold_otsu(ndwi[valid]); water = (ndwi > T_w) & valid
blue = cube[I["blue"]]; T_c = threshold_otsu(np.nan_to_num(blue)[np.isfinite(blue)]); cloud = np.nan_to_num(blue) > T_c
fig, ax = plt.subplots(1, 4, figsize=(20, 4.6))
ax[0].hist(ndwi[valid], bins=200, range=(-1, 1), color="steelblue"); ax[0].axvline(T_w, color="red"); ax[0].set_title(f"NDWI histogram, Otsu T = {T_w:.3f}")
ax[1].imshow(water, cmap="Blues"); ax[1].set_title(f"NDWI > T : water = {100*water[valid].mean():.1f} % of valid")
ax[2].imshow(scl == 6, cmap="Blues"); ax[2].set_title("SCL == 6 (Sen2Cor's water) - a second opinion")
ax[3].imshow(cloud, cmap="gray"); ax[3].set_title(f"blue > {T_c:.3f}: 'cloud' = {100*cloud.mean():.1f} % (bright roofs too!)")
for a in ax[1:]: a.axis("off")
plt.tight_layout(); plt.show()

## 10 · Cleaning a mask: morphology and connected components  *(slide 28)*
Open (remove specks), close (fill holes), label blobs, drop the small ones, measure the rest — an object detector for free.

In [ ]:
from skimage import measure
k3 = np.ones((3, 3), np.uint8)
m = cv2.morphologyEx(water.astype(np.uint8), cv2.MORPH_OPEN, k3); m = cv2.morphologyEx(m, cv2.MORPH_CLOSE, k3)
lab = measure.label(m, connectivity=2); props = [p for p in measure.regionprops(lab) if p.area >= 50]
fig, ax = plt.subplots(1, 3, figsize=(17, 5))
ax[0].imshow(water, cmap="Blues"); ax[0].set_title(f"raw Otsu mask: {measure.label(water, connectivity=2).max()} blobs")
ax[1].imshow(m, cmap="Blues"); ax[1].set_title(f"after open + close: {lab.max()} blobs")
ax[2].imshow(np.where(np.isin(lab, [p.label for p in props]), lab, 0), cmap="nipy_spectral"); ax[2].set_title(f"{len(props)} water bodies >= 50 px (0.5 ha)")
for a in ax: a.axis("off")
plt.tight_layout(); plt.show()
big = sorted(props, key=lambda p: -p.area)[:5]
for p in big: print(f"  blob {p.label:4d}: {int(p.area):8,d} px = {p.area * GSD * GSD / 1e6:7.2f} km2, centroid row {p.centroid[0]:.0f} col {p.centroid[1]:.0f}, bbox {p.bbox}")

## 11 · From rules to learning  *(slide 29)*
Same job, more parameters: Otsu (1 number) vs a random forest on eight hand-made features (10⁴ parameters), both scored against SCL water — with SCL's own errors as label noise.

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from scipy import ndimage
def prf(pred, ref, m):
    p, y = pred[m], ref[m]; tp, fp, fn = (p & y).sum(), (p & ~y).sum(), (~p & y).sum()
    return dict(precision=tp/(tp+fp+1e-9), recall=tp/(tp+fn+1e-9), IoU=tp/(tp+fp+fn+1e-9))
ref = scl == 6
local_std = ndimage.generic_filter(np.nan_to_num(nir), np.std, size=5)
X = np.stack([np.nan_to_num(cube[i]) for i in range(6)] + [np.nan_to_num(ndvi), np.nan_to_num(ndwi), local_std], -1)
H, W = valid.shape; train = valid & (np.arange(W)[None, :] < W // 2); test = valid & ~train     # left half trains, right half tests (a spatial split!)
rng = np.random.default_rng(0); idx = np.flatnonzero(train); idx = rng.choice(idx, min(40000, idx.size), replace=False)
rf = RandomForestClassifier(n_estimators=60, max_depth=12, n_jobs=-1, random_state=0).fit(X.reshape(-1, 9)[idx], ref.ravel()[idx])
rf_pred = rf.predict(X.reshape(-1, 9)).reshape(H, W).astype(bool)
print("scored on the RIGHT half only (never seen in training):")
for name, pred in [("Otsu on NDWI (1 parameter)", ndwi > T_w), ("random forest, 9 features", rf_pred)]:
    print(f"  {name:28s}", {k: round(float(v), 3) for k, v in prf(pred, ref, test).items()})
print("feature importances:", dict(zip(BAND_ORDER + ["ndvi", "ndwi", "nir_std5"], [float(v) for v in np.round(rf.feature_importances_, 2)])))

## 12 · Class work 3 — the cell on slide 31
Run it on **your** window, write down the four numbers, then break it: shift `T` by ±0.1 and mask SCL cloud classes first.

In [ ]:
T = threshold_otsu(ndwi[np.isfinite(ndwi)])
water_pred = ndwi > T                      # 1. your classifier
water_ref  = (scl == 6)                    # 2. a reference label (imperfect!)
tp = (water_pred & water_ref).sum(); fp = (water_pred & ~water_ref).sum(); fn = (~water_pred & water_ref).sum()
print("Otsu T =", round(float(T), 3))
print("precision %.3f  recall %.3f  IoU %.3f" % (tp/(tp+fp), tp/(tp+fn), tp/(tp+fp+fn)))
# --- break it ---
for dT in (-0.1, +0.1):
    p = ndwi > T + dT; tp2, fp2, fn2 = (p & water_ref).sum(), (p & ~water_ref).sum(), (~p & water_ref).sum()
    print(f"  T {T+dT:+.3f}: precision {tp2/(tp2+fp2):.3f} recall {tp2/(tp2+fn2):.3f} IoU {tp2/(tp2+fp2+fn2):.3f}")
p = water_pred & valid; y = water_ref & valid; tp3, fp3, fn3 = (p & y).sum(), (p & ~y).sum(), (~p & y).sum()
print(f"  cloud/nodata masked first: precision {tp3/(tp3+fp3):.3f} recall {tp3/(tp3+fn3):.3f} IoU {tp3/(tp3+fp3+fn3):.3f}")

## 13 · Preview of Lab 1: chips and two splits  *(slides 35, 42, 58)*
Cut 128-px chips with bookkeeping, then colour them by a random split and by a spatial-block split. Lab 1 scores both and explains the gap.

In [ ]:
CH = 128
chips = [(r, c) for r in range(0, H - CH + 1, CH) for c in range(0, W - CH + 1, CH) if valid[r:r+CH, c:c+CH].mean() > 0.7]
rng = np.random.default_rng(0)
split_A = {rc: rng.choice(["train", "val", "test"], p=[.6, .2, .2]) for rc in chips}
block = lambda rc: (rc[0] // (2 * CH), rc[1] // (2 * CH))
blocks = sorted({block(rc) for rc in chips}); rng.shuffle(blocks)
nb = len(blocks); parts = ["train"] * max(1, int(.6 * nb)) + ["val"] * max(1, int(.2 * nb)); parts += ["test"] * (nb - len(parts))
b_split = dict(zip(blocks, parts)); split_B = {rc: b_split[block(rc)] for rc in chips}
col = {"train": "#2D7D5B", "val": "#3066BE", "test": "#B8352B"}
fig, ax = plt.subplots(1, 2, figsize=(14, 6))
for a, sp, t in zip(ax, [split_A, split_B], ["split A: random chips (leaks geography)", "split B: 2 x 2 chip blocks (honest)"]):
    a.imshow(st); a.set_title(t); a.axis("off")
    for (r, c), s in sp.items(): a.add_patch(plt.Rectangle((c, r), CH, CH, fill=True, alpha=0.35, color=col[s], ec="white", lw=0.6))
plt.tight_layout(); plt.show()
print(len(chips), "chips;", {s: sum(v == s for v in split_B.values()) for s in col}, "in split B. Every chip keeps (row, col, tile, date).")

## What to take away
* A satellite image is a **measurement in a file with a contract** — you just read all six sections of it.
* Every classical step (stretch, kernel, threshold, morphology) is a function **you chose**; the labels you made with Otsu are exactly what a network will be trained to predict, errors included.
* Lab 1 (due Sunday 27 Sep 23:59) turns section 13 into a scored dataset with a card. Notebook: `Week3_Lab1_Dataset_Contract.ipynb` in the same repo.

*Course repo: [trongan93/dl-space-2026](https://github.com/trongan93/dl-space-2026) · Sentinel-2 data: Copernicus Sentinel data 2026, via Earth Search (Element 84 / AWS).*